# 📊 Minimum Advertised Price (MAP) Compliance: Data Engineering & ETL Pipeline

**Author**: Derin Girish D G  
**Project**: Minimum Advertised Price (MAP) Policies Compliance & Reseller Monitoring  
**Environment**: Python 3.12 | Pandas | MySQL | Feed Engine  

---

## 🎯 Executive Summary & Business Problem

In multi-channel retail, Original Equipment Manufacturers (OEMs) such as **HP** and **Dell** enforce **Minimum Advertised Price (MAP)** policies to safeguard brand value, preserve retail partner margins, and prevent destructive price erosion across e-commerce marketplaces (Amazon, Flipkart, eBay, Snapdeal).

This notebook implements the foundational **Data Ingestion, Parsing, Transformation, and Normalization Pipeline** that ingests heterogeneous raw files from `data_inputs/`, cleanses taxonomies, calculates the **Lowest Possible Price (LPP)**, resolves reseller alias identities, and exports standardized 3NF relational database tables into `data_processed/`.

---

## 🏗️ End-to-End ETL Pipeline Architecture

### Stage 1: Heterogeneous Raw Inputs (`data_inputs/`)
| Source File | Format | Purpose | Key Attributes |
| :--- | :--- | :--- | :--- |
| `PN_SKU.xml` | XML | Master Product Catalog | Product Number (`PN`), Stock Keeping Unit (`SKU`) |
| `product_info.json` | JSON | Product Taxonomy | Composite line codes (`HP_PC_Laptop`, etc.) |
| `MAP_PL_SKU.yaml` | YAML | Baseline Pricing Floor | Minimum Advertised Price (`MAP`) |
| `lpp_calcuation.xlsx` | Excel | Authorized Margin Matrix | Subcategory discount rates (3.4% - 15.6%) |
| `promotion_sku.xlsx` | Excel | Promotional Rules | Q1 - Q4 promotional discount terms |
| `seller_name.xlsx` | Excel | Entity Resolution Mapping | Marketplace seller aliases $\longrightarrow$ Canonical names |
| `15 Daily CSVs` | CSV | Marketplace Surveillance | Daily feeds from `2025-12-21` to `2026-01-05` (147,900 rows) |

### Stage 2: Core Data Transformation & Normalization Engine
1. **DOM Parsing & Deduplication**: Traverses XML elements, normalizes casing discrepancies, and establishes unique primary keys.
2. **Hierarchical Taxonomy Normalization**: Deconstructs composite codes into `COMPANY` (HP / DELL), `CATEGORY`, and `SUB_CATEGORY`.
3. **Dynamic LPP Calculation**: Computes lowest allowable selling prices:
   $$\text{LPP} = \text{MAP} \times (1 - \text{Margin}_{\text{Sub\_Category}})$$
4. **Entity Homologation**: Resolves 900+ marketplace storefront aliases into canonical parent accounts using lookup mapping.
5. **Multi-Batch Concatenation**: Aggregates 147,900 daily surveillance observations across 5 international regions.

### Stage 3: Relational 3NF Database Tables (`data_processed/database_tables/`)
* **`sku_table.csv`** (1,546 rows): Primary catalog master (`PN`, `SKU` [PK]).
* **`pl_table.csv`** (1,546 rows): Product line taxonomy (`PL`, `SKU`, `SUB_CATEGORY`).
* **`price_list_table.csv`** (1,546 rows): Core pricing baseline (`PL`, `SKU`, `MAP`, `LPP`).
* **`seller_mapping_table.csv`** (900 rows): Entity resolution mapping (`SELLER_NAME` [PK], `HOMOLOGATED_NAME`).
* **`category_mapping_table.csv`** (1,546 rows): Hierarchical taxonomy mapping (`CATEGORY`, `SUB_CATEGORY`, `PL`, `SKU`).
* **`promotion_table.csv`** (96 rows): Active seasonal promotions catalog (`PL`, `SKU`, `SEASON`, `PROMOTION`).
* **`seller_table.csv`** (147,900 rows): Consolidated surveillance feed across all markets.


### 0.1 Environment Setup & Imports

We import standard data engineering libraries and establish strict path definitions anchored relative to the notebook directory (`src/notebooks/`).

In [13]:
import os
import json
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

# Display settings for clear reporting
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Directory configurations
DATA_INPUTS_DIR = Path("../../data_inputs")
DATA_CODE_DIR = Path("../../data_processed")
DATABASE_TABLES_DIR = DATA_CODE_DIR / "database_tables"
INTERMEDIATE_DIR = DATA_CODE_DIR / "intermediate"

DATABASE_TABLES_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")
print("• Raw Data Source:            data_inputs/")
print("• Database Tables Output:    data_processed/database_tables/")
print("• Intermediate Data Output:  data_processed/intermediate/")

Environment initialized successfully.
• Raw Data Source:            data_given/
• Database Tables Output:    data_processed/database_tables/
• Intermediate Data Output:  data_processed/intermediate/


---
## 📦 Stage 1: Product Catalog Ingestion (`PN_SKU.xml`)

### Business Context:
The enterprise ERP system stores the internal **Product Number (PN)** and customer-facing **Stock Keeping Unit (SKU)** in XML format.
* **Goal**: Parse the nested XML tree, construct the canonical SKU catalog table, inspect duplicate SKU assignments, and enforce unique primary keys.


In [14]:
# 1.1 Parse XML Catalog
xml_path = DATA_INPUTS_DIR / "master_catalog" / "PN_SKU.xml"
tree = ET.parse(xml_path)
root = tree.getroot()

records = []
for child in root:
    record = {}
    for subchild in child:
        tag = subchild.tag.strip()
        record[tag] = subchild.text.strip() if subchild.text else None
    records.append(record)

sku_raw = pd.DataFrame(records)
# Standardize XML tag name 'Sku' / 'sku' -> 'SKU'
sku_raw.columns = [col.upper() if col.lower() == "sku" else col for col in sku_raw.columns]

print(f"Extracted {len(sku_raw)} records from XML.")
sku_raw.head()

Extracted 1598 records from XML.


,PN,SKU
0,Pib01a1,0X749QB
1,Pib01a2,75X62BB
2,Pib01a3,DF627B
3,Pib01a4,4ZB96A
4,Pib01a5,0R0N0QB


In [15]:
# 1.2 Data Integrity: Deduplication & Validation
# Ensure column naming robustness
if "SKU" not in sku_raw.columns:
    sku_raw.columns = [col.upper() if col.lower() == "sku" else col for col in sku_raw.columns]

print("Raw SKU Catalog Quality Check:")
print(f"- Total rows:            {len(sku_raw)}")
print(f"- Missing values:       {sku_raw.isna().sum().to_dict()}")
print(f"- Duplicate PN count:   {sku_raw['PN'].duplicated().sum()}")
print(f"- Duplicate SKU count:  {sku_raw['SKU'].duplicated().sum()}")

# In business logic, SKU must serve as the primary key.
# We retain the first valid occurrence of any duplicate SKU.
sku_table_data = sku_raw.drop_duplicates(subset=["SKU"], keep="first").reset_index(drop=True)

# Verification assertion
assert sku_table_data["SKU"].duplicated().sum() == 0, "Primary Key violation: Duplicate SKUs remain!"
print(f"\nCleaned SKU Table: {len(sku_table_data)} unique product SKUs established.")

Raw SKU Catalog Quality Check:
- Total rows:            1598
- Missing values:       {'PN': 0, 'SKU': 0}
- Duplicate PN count:   0
- Duplicate SKU count:  4

Cleaned SKU Table: 1594 unique product SKUs established.


---
## 🏷️ Stage 2: Product Taxonomy & Brand Extraction (`product_info.json`)

### Business Context:
Product line hierarchies arrive in JSON format with composite codes (e.g. `HP_PC_Laptop`).
* **Transformations**:
  1. Standardize column names (`sku` $\longrightarrow$ `SKU`).
  2. Parse the `PL` string to extract `COMPANY` (HP / DELL), `CATEGORY` (PC, Supply, Print Hardware), and `SUB_CATEGORY`.
  3. Filter invalid / corrupted records (`UNK` = Unknown, `SJ` = Scratch placeholder).
  4. Deduplicate SKUs.
  5. Standardize abbreviated subcategory codes into industry-standard names:
     - `TON` $\longrightarrow$ Toner
     - `LJ` $\longrightarrow$ Laserjet
     - `IJ` $\longrightarrow$ Inkjet
     - `INK` $\longrightarrow$ Ink


In [16]:
# 2.1 Load and parse JSON metadata
json_path = DATA_INPUTS_DIR / "master_catalog" / "product_info.json"
product_info = pd.read_json(json_path)
product_info.rename(columns={"sku": "SKU"}, inplace=True)

# 2.2 Extract Taxonomy Attributes from PL string
split_series = product_info["PL"].str.strip().str.split("_")
product_info["COMPANY"] = split_series.str[0].str[:-2]
product_info["CATEGORY"] = split_series.str[-2]
product_info["SUB_CATEGORY"] = split_series.str[-1]

# 2.3 Filter Corrupted / Placeholder Data
initial_count = len(product_info)
product_info = product_info[~product_info["SUB_CATEGORY"].isin(["UNK", "SJ"])].reset_index(drop=True)
print(f"Filtered out placeholder records: {initial_count - len(product_info)} rows dropped.")

# 2.4 Deduplicate on SKU
product_info = product_info.drop_duplicates(subset=["SKU"], keep="first").reset_index(drop=True)

# 2.5 Standardize Subcategory Nomenclature
SUB_CATEGORY_map = {
    "TON": "Toner",
    "LJ": "Laserjet",
    "IJ": "Inkjet",
    "INK": "Ink",
    "Laptop": "Laptop",
    "Monitor": "Monitor",
    "Desktop": "Desktop",
}
product_info["SUB_CATEGORY"] = product_info["SUB_CATEGORY"].replace(SUB_CATEGORY_map)

derived_product_info = product_info.copy()
print(f"Product Taxonomy Table established: {len(derived_product_info)} records.")
derived_product_info[["SKU", "PL", "COMPANY", "CATEGORY", "SUB_CATEGORY"]].head()

Filtered out placeholder records: 48 rows dropped.
Product Taxonomy Table established: 1546 records.


,SKU,PL,COMPANY,CATEGORY,SUB_CATEGORY
0,0X749QB,DELL0X_PC_Laptop,DELL,PC,Laptop
1,75X62BB,DELL75_PC_Monitor,DELL,PC,Monitor
2,DF627B,DELLDF_SUP_TON,DELL,SUP,Toner
3,4ZB96A,HP4Z_PH_LJ,HP,PH,Laserjet
4,0R0N0QB,DELL0R_PC_Laptop,DELL,PC,Laptop


---
## 💲 Stage 3: Baseline MAP Pricing Ingestion (`MAP_PL_SKU.yaml`)

### Business Context:
The client's official Minimum Advertised Price (MAP) schedule is defined in YAML configuration files.
* **Goal**: Parse YAML price matrices, deduplicate SKUs, and merge MAP into the core product catalog.


In [17]:
# 3.1 Load YAML Pricing Configuration
yaml_path = DATA_INPUTS_DIR / "master_catalog" / "MAP_PL_SKU.yaml"
with open(yaml_path, "r", encoding="utf-8") as file:
    yaml_data = yaml.safe_load(file)

product_info_with_map = pd.DataFrame(yaml_data)
product_info_with_map.rename(columns={"sku": "SKU"}, inplace=True)

# Deduplicate on SKU
product_info_with_map = product_info_with_map.drop_duplicates(subset=["SKU"], keep="first").reset_index(drop=True)

# 3.2 Join MAP with Master Catalog
derived_product_info = derived_product_info.merge(
    product_info_with_map[["SKU", "MAP"]], on="SKU", how="inner"
)

# Quality assertion: No null MAP prices
assert derived_product_info["MAP"].isna().sum() == 0, "Missing MAP price encountered after join!"
print(f"MAP prices successfully integrated. Master catalog now contains {len(derived_product_info)} SKUs with MAP.")
derived_product_info[["SKU", "PL", "COMPANY", "SUB_CATEGORY", "MAP"]].head()

MAP prices successfully integrated. Master catalog now contains 1546 SKUs with MAP.


,SKU,PL,COMPANY,SUB_CATEGORY,MAP
0,0X749QB,DELL0X_PC_Laptop,DELL,Laptop,106684
1,75X62BB,DELL75_PC_Monitor,DELL,Monitor,35611
2,DF627B,DELLDF_SUP_TON,DELL,Toner,29914
3,4ZB96A,HP4Z_PH_LJ,HP,Laserjet,39258
4,0R0N0QB,DELL0R_PC_Laptop,DELL,Laptop,139670


---
## 🏢 Stage 4: Reseller Alias Homologation (`seller_name.xlsx`)

### Business Context:
Resellers often sell on Amazon under one legal name (e.g., `Cloudtail`, `Appario`) and on Flipkart or eBay under other trade names. 
* **Homologation**: Mapping diverse marketplace storefront aliases (`SELLER_NAME`) to their canonical parent organization (`HOMOLOGATED_NAME`) so penalties and warnings target the true entity.


In [18]:
# 4.1 Ingest Reseller Mapping
seller_file = DATA_INPUTS_DIR / "master_catalog" / "seller_name.xlsx"
seller_data = pd.read_excel(seller_file)

# Standardize column headers
seller_data = seller_data.rename(
    columns={"Ssellers_name": "SELLER_NAME", "homologated_sellers": "HOMOLOGATED_NAME"}
)

# Strip whitespace and remove placeholder/header artifacts
seller_data["SELLER_NAME"] = seller_data["SELLER_NAME"].astype(str).str.strip()
seller_data["HOMOLOGATED_NAME"] = seller_data["HOMOLOGATED_NAME"].astype(str).str.strip()

seller_data = seller_data[
    seller_data["SELLER_NAME"].str.lower() != "seller"
].drop_duplicates(subset=["SELLER_NAME"], keep="first").reset_index(drop=True)

print(f"Cleaned Seller Mapping Table: {len(seller_data)} unique seller aliases mapped.")
seller_data.head()

Cleaned Seller Mapping Table: 900 unique seller aliases mapped.


,SELLER_NAME,HOMOLOGATED_NAME
0,Cocoblu Retail,BJ's Wholesale Club
1,RK World Infocom Pvt Ltd,American Eagle
2,Appario Retail Private Ltd,Tapestry
3,RetailEZ Pvt Ltd,Wakefern / ShopRite
4,ETrade Online,Harbor Freight Tools


---
## 📉 Stage 5: Dynamic LPP (Lowest Possible Price) Calculation (`lpp_calcuation.xlsx`)

### Business Context:
The **Lowest Possible Price (LPP)** represents the legal minimum price a reseller can advertise before triggering a compliance violation. It is derived from the MAP using brand- and category-specific authorized discount allowances:

$$\text{LPP} = \text{MAP} \times (1 - \text{Discount Rate})$$

Where discount allowances differ between OEMs (HP vs. Dell) across categories (Laptop, Desktop, Ink, Toner, etc.).


In [19]:
# 5.1 Ingest Brand Discount Matrix
lpp_file = DATA_INPUTS_DIR / "master_catalog" / "lpp_calcuation.xlsx"
lpp_data = pd.read_excel(lpp_file, usecols="B:D", skiprows=5, nrows=8)
lpp_data = lpp_data.rename(columns={"Sub_category": "SUB_CATEGORY", "Hp": "HP", "Dell": "DELL"})

# Clean numbered prefixes (e.g. '1. Laptop' -> 'Laptop')
lpp_data["SUB_CATEGORY"] = lpp_data["SUB_CATEGORY"].str.replace(r"^\d+\.\s*", "", regex=True)

# Build nested lookup dictionary: {COMPANY: {SUB_CATEGORY: discount_rate}}
lpp_matrix = lpp_data.set_index("SUB_CATEGORY").to_dict()
print("Authorized Discount Rates:")
print(pd.DataFrame(lpp_matrix))

# 5.2 Vectorized LPP Calculation
def compute_lpp(row):
    company = row["COMPANY"]
    sub_cat = row["SUB_CATEGORY"]
    discount = lpp_matrix.get(company, {}).get(sub_cat, 0.0)
    return round(row["MAP"] * (1.0 - discount), 2)

derived_product_info["LPP"] = derived_product_info.apply(compute_lpp, axis=1)

# Integrity check: LPP must never exceed MAP
assert (derived_product_info["LPP"] <= derived_product_info["MAP"]).all(), "Logic Error: LPP > MAP detected!"
print(f"\nLPP calculated successfully for {len(derived_product_info)} products.")
derived_product_info[["SKU", "PL", "COMPANY", "SUB_CATEGORY", "MAP", "LPP"]].head()

Authorized Discount Rates:
             HP   DELL
Laptop    0.034  0.078
Monitor   0.012  0.056
Toner     0.091  0.023
Laserjet  0.065  0.035
Inkjet    0.012  0.056
Desktop   0.091  0.023
Ink       0.065  0.035

LPP calculated successfully for 1546 products.


,SKU,PL,COMPANY,SUB_CATEGORY,MAP,LPP
0,0X749QB,DELL0X_PC_Laptop,DELL,Laptop,106684,98362.65
1,75X62BB,DELL75_PC_Monitor,DELL,Monitor,35611,33616.78
2,DF627B,DELLDF_SUP_TON,DELL,Toner,29914,29225.98
3,4ZB96A,HP4Z_PH_LJ,HP,Laserjet,39258,36706.23
4,0R0N0QB,DELL0R_PC_Laptop,DELL,Laptop,139670,128775.74


---
## 🎁 Stage 6: Seasonal Promotional Pricing Logic (`promotion_sku.xlsx`)

### Business Context:
During peak shopping quarters (e.g. Q4 holiday season), the client authorizes temporary promotional discounts on select SKUs. Advertised prices below standard LPP during these periods are permitted if they adhere to promotional rules.


In [20]:
# 6.1 Ingest Promotion Rules & Eligibility List
promo_file = DATA_INPUTS_DIR / "master_catalog" / "promotion_sku.xlsx"

# Parse quarterly discount allowances by category
promotion_condition = pd.read_excel(promo_file, usecols="B:F", skiprows=6, nrows=4)
promotion_condition = promotion_condition.rename(
    columns={
        "Category": "CATEGORY",
        "Q1 (Nov-Jan)": "Q1",
        "Q2 (Feb-Apr)": "Q2",
        "Q3 (May-Jul)": "Q3",
        "Q4 (Aug-Oct)": "Q4",
    }
)
promotion_condition["CATEGORY"] = promotion_condition["CATEGORY"].replace(
    {"Supply": "SUP", "Print Hardware": "PH", "PC": "PC"}
)

# Parse list of SKUs approved for seasonal promotions
promotional_skus = pd.read_excel(promo_file, usecols="B", skiprows=15, header=None)
promotional_skus.columns = ["SKU"]
promotional_skus = promotional_skus.dropna().drop_duplicates().reset_index(drop=True)
promotional_skus["SKU"] = promotional_skus["SKU"].astype(str).str.strip()

# 6.2 Construct Consolidated Promotion Schedule for Q4
promotion_data = promotional_skus.merge(
    derived_product_info[["SKU", "PL", "CATEGORY"]], on="SKU", how="inner"
)
promotion_data["SEASON"] = "Q4"

# Map Q4 authorized promotional discount rate
q4_lookup = promotion_condition.set_index("CATEGORY")["Q4"].to_dict()
promotion_data["PROMOTION"] = promotion_data["CATEGORY"].map(q4_lookup)

print(f"Established {len(promotion_data)} active promotional SKU rules for Season Q4.")
promotion_data.head()

Established 96 active promotional SKU rules for Season Q4.


,SKU,PL,CATEGORY,SEASON,PROMOTION
0,G0L15B,DELLG0_SUP_INK,SUP,Q4,20% off ink
1,7Y4T6QB,DELL7Y_PC_Desktop,PC,Q4,15 % off
2,4K0V8PA,HP4K_SUP_INK,SUP,Q4,20% off ink
3,385Z7PA,HP38_PC_Desktop,PC,Q4,15 % off
4,T6M05AA,HPT6_SUP_INK,SUP,Q4,20% off ink


---
## 🏛️ Stage 7: Relational Schema Normalization (3NF)

### Business Context:
To prepare for relational database loading into MySQL (`MAP_PROJECT`), we project the cleaned dataframes into dedicated, normalized entities with distinct primary and foreign key responsibilities:

1. **`SKU`**: Catalog master (`PN`, `SKU` [PK]).
2. **`PL`**: Product line categorization (`PL`, `SKU` [PK: PL, SKU], `SUB_CATEGORY`).
3. **`PRICE_LIST`**: Price policy baseline (`PL`, `SKU` [PK: PL, SKU], `MAP`, `LPP`).
4. **`SELLER_MAPPING`**: Entity resolution mapping (`SELLER_NAME` [PK], `HOMOLOGATED_NAME`).
5. **`CATEGORY_MAPPING`**: Category taxonomy (`CATEGORY`, `SUB_CATEGORY`, `PL`, `SKU` [PK]).
6. **`PROMOTION`**: Active seasonal discounts (`PL`, `SKU`, `SEASON` [PK: SKU, SEASON], `PROMOTION`).


In [21]:
# 7.1 Construct Relational Tables
sku_table = sku_table_data[["PN", "SKU"]].copy()

pl_table = derived_product_info[["PL", "SKU", "SUB_CATEGORY"]].copy()

price_list_table = derived_product_info[["PL", "SKU", "MAP", "LPP"]].copy()

seller_mapping_table = seller_data[["SELLER_NAME", "HOMOLOGATED_NAME"]].copy()

category_mapping_table = derived_product_info[
    ["CATEGORY", "SUB_CATEGORY", "PL", "SKU"]
].copy()

promotion_table = promotion_data[["PL", "SKU", "SEASON", "PROMOTION"]].copy()

print("Relational entities created:")
print(f"- SKU Table:              {len(sku_table)} rows")
print(f"- PL Table:               {len(pl_table)} rows")
print(f"- PRICE_LIST Table:       {len(price_list_table)} rows")
print(f"- SELLER_MAPPING Table:   {len(seller_mapping_table)} rows")
print(f"- CATEGORY_MAPPING Table: {len(category_mapping_table)} rows")
print(f"- PROMOTION Table:        {len(promotion_table)} rows")

Relational entities created:
- SKU Table:              1594 rows
- PL Table:               1546 rows
- PRICE_LIST Table:       1546 rows
- SELLER_MAPPING Table:   900 rows
- CATEGORY_MAPPING Table: 1546 rows
- PROMOTION Table:        96 rows


---
## 🌐 Stage 8: Marketplace Surveillance Feeds Ingestion (Task 3 Daily Seller Data)

### Business Context:
Every day, automated monitoring collects pricing snapshots across Amazon, Flipkart, eBay, and Snapdeal.
* **Volume**: 15 consecutive daily batches (`2025-12-21` through `2026-01-05`), totaling ~148,000 market observations.
* **Transformations**:
  1. Iterate across all 15 daily CSV feeds.
  2. Dynamically extract collection date (`Last time collected`) from each filename.
  3. Standardize column names to conform with the Data Dictionary schema:
     `['Seller_name', 'SKU', 'Region', 'Advertised_price', 'Marketplace', 'Screenshot', 'Live_link', 'Last time collected']`.
  4. Ensure graceful handling of nulls for historical batch links/screenshots.


In [22]:
# 8.1 Batch Ingestion of Daily Seller Feeds
seller_data_folder = DATA_INPUTS_DIR / "marketplace_feeds" / "daily_seller_feeds"
seller_files = sorted(seller_data_folder.glob("*.csv"))

print(f"Ingesting {len(seller_files)} daily seller feed batches from {seller_data_folder.name}/...")

seller_dfs = []
for file in seller_files:
    df = pd.read_csv(file)
    # Extract date string from filename (e.g. '2025-12-21')
    df["Last time collected"] = file.stem
    seller_dfs.append(df)

seller_table_raw = pd.concat(seller_dfs, ignore_index=True)
print(f"Total raw surveillance records concatenated: {len(seller_table_raw)}")

# 8.2 Schema Alignment with Data Dictionary
seller_table = seller_table_raw.rename(
    columns={
        "region": "Region",
        "adv_price": "Advertised_price",
        "live_link": "Live_link",
    }
)

# Ensure audit columns exist and handle nulls
seller_table["Screenshot"] = seller_table.get("Screenshot", pd.Series(dtype=str)).fillna("")
seller_table["Live_link"] = seller_table.get("Live_link", pd.Series(dtype=str)).fillna("")

# Order columns according to official schema specification
columns_order = [
    "Seller_name",
    "SKU",
    "Region",
    "Advertised_price",
    "Marketplace",
    "Screenshot",
    "Live_link",
    "Last time collected",
]
seller_table = seller_table[columns_order]

print(f"\nConsolidated Seller Table: {len(seller_table)} records across {seller_table['Last time collected'].nunique()} collection days.")
seller_table.head()

Ingesting 15 daily seller feed batches from seller_data/...
Total raw surveillance records concatenated: 147900

Consolidated Seller Table: 147900 records across 15 collection days.


,Seller_name,SKU,Region,Advertised_price,Marketplace,Screenshot,Live_link,Last time collected
0,H.R.&COMPANY INDIA-LIMESHOT™ (100% Satisfaction ),6X3W4PA,USA,460.440,Flipkart,,,2025-12-21
1,SHOP 24/7,81H48PA,USA,940.200,Flipkart,,,2025-12-21
2,OPSUZY,676M7A,UAE,651.640,eBay,,,2025-12-21
3,Abbey Edge,CF361A,UK,202.382,eBay,,,2025-12-21
4,ASTraders,P2V25A,UAE,368.280,Snapdeal,,,2025-12-21


---
## 💾 Stage 9: Pipeline Export & Enterprise Data Quality Scorecard

### Architectural Separation of Output Data:
In enterprise data engineering, we strictly distinguish between:
1. **Production Relational Tables (`data_code/database_tables/`)**: Normalized 3NF tables loaded directly into MySQL (`SKU`, `PL`, `PRICE_LIST`, `SELLER_MAPPING`, `CATEGORY_MAPPING`, `PROMOTION`, and `SELLER`).
2. **Intermediate Transformation Artifacts (`data_code/intermediate/`)**: Working tables and reference matrices generated during data processing (`derived_product_info`, `lpp_data`, `promotion_condition`, `promotional_skus`).


In [23]:
# 9.1 Export Production Database Relational Tables (for MySQL)
db_exports = {
    "sku_table.csv": sku_table,
    "pl_table.csv": pl_table,
    "price_list_table.csv": price_list_table,
    "seller_mapping_table.csv": seller_mapping_table,
    "category_mapping_table.csv": category_mapping_table,
    "promotion_table.csv": promotion_table,
    "seller_table.csv": seller_table,
}

for filename, df in db_exports.items():
    df.to_csv(DATABASE_TABLES_DIR / filename, index=False)

print(f"Exported {len(db_exports)} relational tables to: data_processed/database_tables/")

# 9.2 Export Intermediate Transformation Artifacts
intermediate_exports = {
    "derived_product_info.csv": derived_product_info,
    "lpp_data.csv": lpp_data,
    "promotion_condition.csv": promotion_condition,
    "promotional_skus.csv": promotional_skus,
}

for filename, df in intermediate_exports.items():
    df.to_csv(INTERMEDIATE_DIR / filename, index=False)

print(f"Exported {len(intermediate_exports)} intermediate artifacts to: data_processed/intermediate/")

Exported 7 relational tables to: data_processed/database_tables/
Exported 4 intermediate artifacts to: data_processed/intermediate/


In [24]:
# 9.3 Enterprise Data Quality & Relational Audit Scorecard
scorecard = []

# Audit Database Tables
for name, df in db_exports.items():
    scorecard.append({
        "Category": "Database Table (MySQL)",
        "Table Name": name.replace(".csv", ""),
        "Row Count": len(df),
        "Column Count": len(df.columns),
        "Missing Values": df.isna().sum().sum(),
        "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 1),
    })

# Audit Intermediate Artifacts
for name, df in intermediate_exports.items():
    scorecard.append({
        "Category": "Intermediate Artifact",
        "Table Name": name.replace(".csv", ""),
        "Row Count": len(df),
        "Column Count": len(df.columns),
        "Missing Values": df.isna().sum().sum(),
        "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 1),
    })

scorecard_df = pd.DataFrame(scorecard)
print("==================================================================")
print("             ENTERPRISE DATA QUALITY AUDIT SCORECARD              ")
print("==================================================================")
scorecard_df

             ENTERPRISE DATA QUALITY AUDIT SCORECARD              


,Category,Table Name,Row Count,Column Count,Missing Values,Memory (KB)
0,Database Table (MySQL),sku_table,1594,2,0,177.4
1,Database Table (MySQL),pl_table,1546,3,0,260.6
2,Database Table (MySQL),price_list_table,1546,4,0,202.9
3,Database Table (MySQL),seller_mapping_table,900,2,0,113.2
4,Database Table (MySQL),category_mapping_table,1546,4,0,338.3
5,Database Table (MySQL),promotion_table,96,4,0,21.6
6,Database Table (MySQL),seller_table,147900,8,0,56812.0
7,Intermediate Artifact,derived_product_info,1546,7,0,441.0
8,Intermediate Artifact,lpp_data,7,3,0,0.6
9,Intermediate Artifact,promotion_condition,3,5,0,1.0
